PASO 1: Configurar PySpark en Colab

In [8]:
# Celda 1: Instalar PySpark
!pip install pyspark findspark -q
print("PySpark instalado!")

PySpark instalado!


PASO 2: Word Count básico

In [9]:
# Celda 2: Word Count básico
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("WordCount_TuNombre") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("ERROR")

# TU TEXTO: Usa párrafos sobre la industria de tu proyecto
texto_negocio = """
[AQUI ESCRIBE 5-10 oraciones sobre la industria de tu proyecto de Big Data.
Ejemplo si tu proyecto es retail:
Las ventas del retail peruano crecieron 15% en 2024. El comercio electrónico
representa el 20% de las ventas totales. Los supermercados lideran las ventas online.
La digitalización del retail es una tendencia imparable en Peru y Latinoamerica.]
"""

# Procesar con MapReduce
palabras_rdd = sc.parallelize([texto_negocio]) \
    .flatMap(lambda x: x.split()) \
    .map(lambda w: (w.lower().strip('.,!?'), 1)) \
    .reduceByKey(lambda a, b: a + b) \
    .filter(lambda x: len(x[0]) > 3)  # filtrar palabras muy cortas

resultado = sorted(palabras_rdd.collect(), key=lambda x: -x[1])

print("=== TOP 20 PALABRAS MÁS FRECUENTES ===")
print(f"{'Palabra':<20} {'Frecuencia':>10}")
print("-" * 32)
for palabra, freq in resultado[:20]:
    print(f"{palabra:<20} {freq:>10}")

=== TOP 20 PALABRAS MÁS FRECUENTES ===
Palabra              Frecuencia
--------------------------------
ventas                        3
proyecto                      2
retail                        2
[aqui                         1
5-10                          1
sobre                         1
ejemplo                       1
peruano                       1
crecieron                     1
representa                    1
lideran                       1
online                        1
tendencia                     1
peru                          1
escribe                       1
oraciones                     1
industria                     1
data                          1
retail:                       1
2024                          1


PASO 3: Análisis de un dataset real con Spark SQL

In [10]:
# Celda 3: Análisis con Spark SQL
import pandas as pd
import numpy as np

# Crear dataset de ventas (simula datos de tu empresa)
np.random.seed(42)
n = 100000  # 100K registros — ya es interesante para Spark

datos = {
    'fecha': pd.date_range('2024-01-01', periods=n, freq='min').astype(str),
    'producto': np.random.choice(['ProductoA', 'ProductoB', 'ProductoC', 'ProductoD'], n),
    'region': np.random.choice(['Lima', 'Arequipa', 'Trujillo', 'Cusco'], n, p=[0.5, 0.2, 0.2, 0.1]),
    'cantidad': np.random.randint(1, 50, n),
    'precio': np.random.uniform(10, 500, n).round(2),
    'canal': np.random.choice(['Online', 'Tienda', 'App_Movil'], n, p=[0.4, 0.4, 0.2])
}

pdf = pd.DataFrame(datos)
pdf['monto'] = pdf['cantidad'] * pdf['precio']

# Convertir a Spark DataFrame
df_spark = spark.createDataFrame(pdf)
df_spark.createOrReplaceTempView("ventas")

print(f"Dataset: {df_spark.count():,} registros cargados en Spark")
print(f"Schema:")
df_spark.printSchema()
# Celda 4: Queries con Spark SQL
print("=== ANÁLISIS 1: Ventas totales por región ===")
spark.sql("""
    SELECT region,
           COUNT(*) as num_transacciones,
           SUM(monto) as monto_total,
           AVG(monto) as ticket_promedio
    FROM ventas
    GROUP BY region
    ORDER BY monto_total DESC
""").show()

print("=== ANÁLISIS 2: Top 3 productos por canal ===")
spark.sql("""
    SELECT canal, producto, SUM(monto) as ventas
    FROM ventas
    GROUP BY canal, producto
    ORDER BY canal, ventas DESC
""").show(12)

print("=== ANÁLISIS 3: Ventas por hora del día ===")
spark.sql("""
    SELECT HOUR(CAST(fecha AS TIMESTAMP)) as hora,
           COUNT(*) as transacciones
    FROM ventas
    GROUP BY hora
    ORDER BY hora
""").show()

Dataset: 100,000 registros cargados en Spark
Schema:
root
 |-- fecha: string (nullable = true)
 |-- producto: string (nullable = true)
 |-- region: string (nullable = true)
 |-- cantidad: long (nullable = true)
 |-- precio: double (nullable = true)
 |-- canal: string (nullable = true)
 |-- monto: double (nullable = true)

=== ANÁLISIS 1: Ventas totales por región ===
+--------+-----------------+--------------------+------------------+
|  region|num_transacciones|         monto_total|   ticket_promedio|
+--------+-----------------+--------------------+------------------+
|    Lima|            49732| 3.173872146100008E8| 6381.951552521531|
|Trujillo|            20325|1.2969851287999982E8| 6381.230646002451|
|Arequipa|            20078|1.2908325811000001E8|6429.0894566191855|
|   Cusco|             9865| 6.297448409000007E7| 6383.627378611259|
+--------+-----------------+--------------------+------------------+

=== ANÁLISIS 2: Top 3 productos por canal ===
+---------+---------+----------

PASO 4: Comparar velocidad Pandas vs. Spark

In [11]:
# Celda 5: Benchmark — esto muestra el VALOR de Spark
import time

# Con Pandas
start = time.time()
resultado_pandas = pdf.groupby('region')['monto'].sum().sort_values(ascending=False)
tiempo_pandas = time.time() - start
print(f"Pandas: {tiempo_pandas*1000:.2f} ms")

# Con Spark
start = time.time()
resultado_spark = df_spark.groupBy('region').sum('monto').orderBy('sum(monto)', ascending=False)
resultado_spark.collect()  # forzar ejecución
tiempo_spark = time.time() - start
print(f"Spark: {tiempo_spark*1000:.2f} ms")

print(f"\nNota: Con 100K registros Pandas puede ser más rápido.")
print(f"Con 100M registros Spark sería 10-100x más rápido porque distribuye el trabajo.")
print(f"Esta es la clave de Big Data: escala a donde Pandas no puede llegar.")

Pandas: 11.69 ms
Spark: 1444.67 ms

Nota: Con 100K registros Pandas puede ser más rápido.
Con 100M registros Spark sería 10-100x más rápido porque distribuye el trabajo.
Esta es la clave de Big Data: escala a donde Pandas no puede llegar.
